In [7]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

documents = load_faq_data()
index = build_index(documents)

In [19]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=messages
)

response.choices[0].message.content

'That\'s great! To help you figure out if you can join the course, I need a little more information. Please tell me:\n\n*   **What is the name of the course?** (e.g., "Introduction to Python Programming," "Advanced Calculus," "Beginner\'s Yoga")\n*   **Where did you discover it?** (e.g., a university website, an online learning platform like Coursera/edX/Udemy, a community center, a company\'s internal training portal)\n*   **Do you know if it\'s currently being offered?** (e.g., is there a start date listed? Is enrollment open?)\n\nOnce I have this information, I can give you a much more specific answer and help you figure out the next steps!'

In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [14]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=messages,
    tools=[search_tool]
)

response.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-2474134539213382211', function=Function(arguments='{"query":"how to join the course"}', name='search'), type='function')]))

In [15]:
import json

call = response.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [16]:
messages.append(response.choices[0].message)

messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json
})

In [18]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=messages,
    tools=[search_tool]
)

response.choices[0].message.content

'Yes, you can still join the course. If you wish to receive a certificate, ensure that you submit your project before the submission deadline.'